# WMDP and Dual-Use Capability Evaluation: Interactive Visual Explorer

> Li et al., "The WMDP Benchmark: Measuring and Reducing Malicious Use With Unlearning" (ICML 2024, arXiv:2403.03218). 4,157 multiple-choice questions across biosecurity (1,520), cybersecurity (2,225), and chemistry (412). Questions operate in the "yellow zone" — proximate enabling knowledge, filtered by multi-expert review and ITAR/EAR legal compliance. Dual purpose: proxy evaluation of dual-use capability, and unlearning benchmark (the companion RMU method reduces WMDP performance while preserving general capability). 2024-2025 field narrative: early OpenAI/Anthropic 2024 evaluations reported "mild uplift" over internet search; by April 2025, OpenAI's Preparedness Framework v2 said models are "on the cusp of meaningfully helping novices create known biological threats." Anthropic's bioweapon-acquisition trial showed 2.53x uplift, insufficient to rule out ASL-3.

Welcome to the interactive companion notebook for **WMDP and Dual-Use Capability Evaluation**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""WMDP-shaped evaluation harness — stdlib Python.

Mock model is a 3-domain expert with per-domain accuracy vectors.
Simulates a WMDP-style multiple-choice evaluation across bio, cyber, chem.
Demonstrates the RMU-style unlearning trade-off: suppress domain-specific
capability, measure the general-capability cost.

Usage: python3 code/main.py
"""

from __future__ import annotations

import random

random.seed(47)


In [ ]:
DOMAINS = {
    "biosecurity":   {"n_questions": 200, "accuracy": 0.72},
    "cybersecurity": {"n_questions": 200, "accuracy": 0.80},
    "chemistry":     {"n_questions": 200, "accuracy": 0.64},
    "mmlu_general":  {"n_questions": 200, "accuracy": 0.78},
}

def evaluate(model_accuracy: dict) -> dict:
    """Run the toy WMDP-shaped benchmark. Returns per-domain score."""
    results = {}
    for domain, cfg in DOMAINS.items():
        correct = 0
        for _ in range(cfg["n_questions"]):
            acc = model_accuracy.get(domain, cfg["accuracy"])
            if random.random() < acc:
                correct += 1
        results[domain] = correct / cfg["n_questions"]
    return results


In [ ]:
def apply_rmu_style_unlearning(model_accuracy: dict,
                               targets: list[str],
                               strength: float = 0.9,
                               collateral: float = 0.03) -> dict:
    """Unlearning intervention: reduce target-domain accuracy by `strength`,
    leak `collateral` accuracy loss to other domains (general capability)."""
    new = dict(model_accuracy)
    for d in targets:
        new[d] = max(0.25, new[d] * (1 - strength))
    for d in new:
        if d not in targets:
            new[d] = max(0.0, new[d] - collateral)
    return new


In [ ]:
def baseline_model() -> dict:
    return {d: cfg["accuracy"] for d, cfg in DOMAINS.items()}

def report(title: str, r: dict) -> None:
    print(f"\n{title}")
    for d, score in r.items():
        print(f"  {d:18s} : {score:.3f}")

def main() -> None:
    print("=" * 70)
    print("WMDP-SHAPED EVALUATION HARNESS (Phase 18, Lesson 17)")
    print("=" * 70)

base = baseline_model()
    report("baseline model accuracy by domain", base)
    baseline_results = evaluate(base)
    report("measured scores (pre-unlearning)", baseline_results)


In [ ]:
# Unlearn bio + chem.
    post = apply_rmu_style_unlearning(base, targets=["biosecurity", "chemistry"],
                                       strength=0.85, collateral=0.04)
    post_results = evaluate(post)
    report("measured scores (post-unlearning: bio + chem)", post_results)

print("\nuplift-style calculation (novice baseline ~= 0.25 random):")
    novice = 0.25
    for d in ("biosecurity", "cybersecurity", "chemistry"):
        pre = baseline_results[d]
        pst = post_results[d]
        uplift_pre = pre / novice
        uplift_post = pst / novice
        print(f"  {d:18s}  pre={uplift_pre:.2f}x novice  post={uplift_post:.2f}x novice")


In [ ]:
print("\n" + "=" * 70)
    print("TAKEAWAY: WMDP gives a per-domain capability number without eliciting")
    print("harmful output. RMU-style unlearning reduces target-domain scores")
    print("with ~3-4% general-capability collateral damage. the 2025 field")
    print("narrative is 'mild uplift' -> 'on the cusp' -> 'insufficient to")
    print("rule out ASL-3' -- each transition backed by a different study.")
    print("=" * 70)


In [ ]:
if __name__ == "__main__":
    main()
